In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Tampilkan lebar kolom penuh (kalau kolomnya panjang)
pd.set_option('display.max_colwidth', None)
# pd.set_option('display.max_rows', None)

In [ ]:
df = pd.read_csv('/kaggle/input/maybanktm/DataMaybank3.csv')

In [ ]:
df_tm = df.copy()

In [ ]:
df.head()

# Cleaning

In [ ]:
import re
import emoji

def clean_text(text: str) -> str:
    # Lowercase
    text = text.lower()
    
    # Hapus URL
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Hapus mention (@username) dan hashtag (#tag)
    text = re.sub(r'@\w+|#\w+', '', text)
    
    # Hapus emoji (via package emoji)
    text = emoji.replace_emoji(text, replace='')
    
    # Hapus emoticon (smiley biasa)
    emoticon_pattern = r'[:;=8][\-o\*\']?[\)\]\(\[dDpP/\:\}\{@\|\\]'
    text = re.sub(emoticon_pattern, '', text)

    # Gabungkan partikel terpisah (contoh: 'service nya' → 'servicenya')
    text = re.sub(r'\b(\w+)\s+(nya|ku|mu)\b', r'\1\2', text)
    
    # Hapus ekstra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Hapus angka
    text = re.sub(r'\d+', '', text)

    # Normalisasi tanda baca berulang (misal ..., !!!, ???)
    text = re.sub(r'([!?.,])\1{1,}', r'\1', text)

    # Normalisasi huruf berulang (contoh: peliittt → pelit)
    text = re.sub(r'(.)\1{1,}', r'\1', text)
    
    return text

# Terapkan ke DataFrame
df['komentar'] = df['komentar'].astype(str).apply(clean_text)

In [ ]:
# Define normalization dictionary
normalize_dict = {
    'ga': 'tidak',
    'gk': 'tidak',
    'ngga': 'tidak',
    'tdk': 'tidak',
    'klo': 'kalau',
    'kalo': 'kalau',
    'jgn': 'jangan',
    'jd': 'jadi',
    'sgt': 'sangat',
    'dlm': 'dalam',
    'jg': 'juga',
    'dgn': 'dengan',
    'yg': 'yang',
    'sbg': 'sebagai',
    'dpt': 'dapat',
    'krn': 'karena',
    'sy': 'saya',
    'km': 'kamu',
    'udh': 'sudah',
    'udah': 'sudah',
    'usah': 'udah',
    'blm': 'belum',
    'bgt': 'banget',
    'cs': 'customer service',
    'lambat': 'lama',
    'sampe': 'sampai',
    'terimakasih': 'terima kasih',
    'gtu': 'begitu',
    'bngt': 'banget',
    'nga': 'tidak',
    'god': 'good',
    'beter': 'better',
    'kren': 'keren',
    'nungu': 'nunggu'
}
# Function to normalize text based on the dictionary
def normalize_text(text):
    text = str(text).lower()
    tokens = text.split()
    normalized_tokens = [normalize_dict.get(token, token) for token in tokens]
    return ' '.join(normalized_tokens)

# Apply normalization and insert as a new column
df['komentar'] = df['komentar'].apply(normalize_text)

In [ ]:
df

In [ ]:
# Subset berdasarkan sentimen
df_positif = df[df['sentimen'] == 'positif'].copy()
df_netral  = df[df['sentimen'] == 'netral'].copy()
df_negatif = df[df['sentimen'] == 'negatif'].copy()


print("Jumlah komentar positif:", len(df_positif))
print("Jumlah komentar netral :", len(df_netral))
print("Jumlah komentar negatif:", len(df_negatif))

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

# Stopword Inggris dari NLTK
stopwords_en = set(stopwords.words('english'))

# Stopword Indonesia dari NLTK
stopwords_id = set(stopwords.words('indonesian'))

# Gabungkan
combined_stopwords = stopwords_en.union(stopwords_id)

In [ ]:
def remove_stopwords(text, stopword_set):
    return ' '.join([word for word in text.split() if word not in stopword_set])

In [ ]:
# Dictionary untuk akses dan modifikasi dinamis
datasets = {
    'positif': df_positif,
    'netral': df_netral,
    'negatif': df_negatif
}

# Proses hapus stopword untuk tiap DataFrame
for label, df_sentimen in datasets.items():
    df_sentimen['komentar_clean'] = df_sentimen['komentar'].apply(lambda x: remove_stopwords(x, combined_stopwords))

In [ ]:
df_positif.to_csv('komen positif.csv', index=False)
df_netral.to_csv('komen netral.csv', index=False)
df_negatif.to_csv('komen negatif.csv', index=False)

# EDA buat dashboard

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Gaya visualisasi profesional
plt.style.use('fivethirtyeight')
plt.rcParams['lines.linewidth'] = 1.5
professional_style = {
    'figure.facecolor': '#1e1e2f',
    'axes.facecolor': '#1e1e2f',
    'savefig.facecolor': '#1e1e2f',
    'axes.grid': True,
    'axes.spines.left': False,
    'axes.spines.right': False,
    'axes.spines.top': False,
    'axes.spines.bottom': False,
    'grid.color': '#333b55',
    'grid.linewidth': 1,
    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'font.size': 13,
    'font.family': 'DejaVu Sans'  # Bisa juga 'Arial', 'Lato', dll
}
plt.rcParams.update(professional_style)

# Warna tone-down profesional
color_map = {'positif': '#5cb85c', 'netral': '#999999', 'negatif': '#d9534f'}

In [ ]:
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter
from itertools import islice

In [ ]:
def plot_word_count_distribution(df, title, color='steelblue'):
    word_counts = df['komentar'].apply(lambda x: len(x.split()))
    plt.figure(figsize=(8, 4))
    sns.histplot(word_counts, bins=20, kde=True, color=color)
    plt.title(f'Distribusi Jumlah Kata - {title}', fontsize=14, weight='bold')
    plt.xlabel('Jumlah Kata per Komentar')
    plt.ylabel('Frekuensi')
    plt.tight_layout()
    plt.show()

def plot_char_count_distribution(df, title, color='mediumpurple'):
    char_counts = df['komentar'].apply(len)
    plt.figure(figsize=(8, 4))
    sns.histplot(char_counts, bins=20, kde=True, color=color)
    plt.title(f'Distribusi Panjang Kalimat - {title}', fontsize=14, weight='bold')
    plt.xlabel('Jumlah Karakter per Komentar')
    plt.ylabel('Frekuensi')
    plt.tight_layout()
    plt.show()


def generate_wordcloud(text_series, title, color='white'):
    text = ' '.join(text_series)
    wc = WordCloud(width=800, height=400, background_color='black', colormap=color).generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(title, fontsize=16, weight='bold')
    plt.tight_layout()
    plt.show()

def get_top_ngrams(corpus, ngram_range=(2, 2), n=None):
    from sklearn.feature_extraction.text import CountVectorizer
    vec = CountVectorizer(ngram_range=ngram_range).fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0) 
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    return words_freq[:n]

def plot_top_ngrams(corpus, title, ngram_range=(2,2), top_n=5, color='skyblue'):
    import matplotlib.pyplot as plt
    import seaborn as sns

    top_ngrams = get_top_ngrams(corpus, ngram_range=ngram_range, n=top_n)
    ngrams = [i[0] for i in top_ngrams]
    freqs = [i[1] for i in top_ngrams]

    plt.figure(figsize=(8, 4))
    ax = sns.barplot(x=freqs, y=ngrams, color=color)

    # Tambahkan label angka di ujung bar
    for i, v in enumerate(freqs):
        plt.text(
            v + 0.5,  # posisi x
            i,        # posisi y = indeks
            str(v),   # isi label
            color='white',
            va='center',
            fontsize=10,
            weight='bold'
        )

    plt.title(title, fontsize=14, weight='bold')
    plt.xlabel('Frekuensi')
    plt.ylabel('Frasa')
    plt.tight_layout()
    plt.show()

def plot_top_locations(df, column='letak', title='Lokasi Terbanyak', top_n=5, color='#5cb85c'):
    import matplotlib.pyplot as plt

    # Hitung lokasi terbanyak
    lokasi_counts = df[column].value_counts().head(top_n)

    # Cek jika kosong
    if lokasi_counts.empty:
        print("⚠️ Data kosong, tidak ada lokasi untuk ditampilkan.")
        return []

    # Plot bar
    plt.figure(figsize=(8, 4))
    ax = lokasi_counts.plot(kind='bar', color=color)

    plt.title(title, fontsize=14, weight='bold')
    plt.xlabel('Letak')
    plt.ylabel('Jumlah')
    plt.xticks(rotation=30, ha='right', fontsize=10)
    plt.yticks(fontsize=10)

    # Tambahkan label jumlah di atas batang
    for i, value in enumerate(lokasi_counts.values):
        y_offset = 0.3 if value < 10 else 2
        label_color = 'white'
        plt.text(
            x=i,
            y=value + y_offset,
            s=str(value),
            ha='center',
            va='bottom',
            fontsize=10,
            color=label_color,
            weight='bold'
        )

    plt.tight_layout()
    plt.show()

## Distribusi Data Sentimen

In [ ]:
sentimen_counts = df['sentimen'].value_counts().sort_index()
sentimen_counts = sentimen_counts.reindex(['positif', 'negatif', 'netral'])

# Bar Chart
plt.figure(figsize=(6, 4))
ax = sns.barplot(
    x=sentimen_counts.index,
    y=sentimen_counts.values,
    palette=color_map
)

plt.title('Distribusi Sentimen', fontsize=16, weight='bold')
plt.xlabel('Sentimen', fontsize=13)
plt.ylabel('Jumlah', fontsize=13)
plt.xticks(fontsize=11)
plt.yticks(fontsize=11)

# Tambahkan label jumlah di atas setiap bar
for i, value in enumerate(sentimen_counts.values):
    plt.text(
        x=i, 
        y=value + 20,  # agak di atas bar
        s=str(value), 
        ha='center', 
        va='bottom', 
        color='white',
        fontsize=12,
        weight='bold'
    )

plt.tight_layout()
plt.show()


# Pie Chart
plt.figure(figsize=(6, 6))
plt.pie(
    sentimen_counts.values,
    labels=sentimen_counts.index,
    autopct='%1.1f%%',
    colors=[color_map[s] for s in sentimen_counts.index],
    textprops={'color': 'white', 'fontsize': 12}
)
plt.title('Proporsi Sentimen', fontsize=16, weight='bold')
plt.tight_layout()
plt.show()

## Positif

### Distribusi Kata

In [ ]:
plot_word_count_distribution(df_positif, title='Positif', color='#5cb85c')

In [ ]:
plot_char_count_distribution(df_positif, title='Positif', color='#5cb85c')

### WordCloud

In [ ]:
# Wordcloud
generate_wordcloud(df_positif['komentar_clean'], title='Wordcloud Komentar Positif', color='Greens')

### Top Bigram

In [ ]:
# Top bigram
plot_top_ngrams(df_positif['komentar_clean'], title='Frasa Populer Komentar Positif', color='#5cb85c')

### Lokasi 

In [ ]:
plot_top_locations(
    df_positif,
    title='Lokasi Terbanyak (Komentar Positif)',
    color='#5cb85c'
)

## Netral

### Distribusi Kata

In [ ]:
plot_word_count_distribution(df_netral, title='Netral', color='#999999')

In [ ]:
plot_char_count_distribution(df_netral, title='Netral', color='#999999')

### WordCloud

In [ ]:
generate_wordcloud(df_netral['komentar_clean'], title='Wordcloud Komentar Netral', color='gray')

### Top Bigram

In [ ]:
plot_top_ngrams(df_netral['komentar_clean'], title='Frasa Populer Komentar Netral', color='#999999')

In [ ]:
plot_top_locations(
    df_netral,
    title='Lokasi Terbanyak (Komentar Netral)',
    color='#999999'
)

## Negatif

## Distribusi Kata

In [ ]:
plot_word_count_distribution(df_negatif, title='Negatif', color='#d9534f')

In [ ]:
plot_char_count_distribution(df_negatif, title='Negatif', color='#d9534f')

### WordCloud

In [ ]:
generate_wordcloud(df_negatif['komentar_clean'], title='Wordcloud Komentar Negatif', color='Reds')

### Top Bigram

In [ ]:
plot_top_ngrams(df_negatif['komentar_clean'], title='Frasa Populer Komentar Negatif', color='#d9534f')

### Kantor

In [ ]:
plot_top_locations(
    df_negatif,
    title='Lokasi Terbanyak (Komentar Negatif)',
    color='#d9534f'
)

# Topic Modeling

In [ ]:
!pip install bertopic sentence-transformers

In [ ]:
df_tm

In [ ]:
df_tm['komentar'] = df_tm['komentar'].apply(normalize_text)

In [ ]:
df_tm

In [ ]:
df_tm['komentar'] = df_tm['komentar'].astype(str).apply(clean_text)

In [ ]:
df_tm

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# Gunakan model multilingual yang efisien
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

In [ ]:
# Komentar per sentimen
komentar_positif = df_positif['komentar'].tolist()
komentar_netral  = df_netral['komentar'].tolist()
komentar_negatif = df_negatif['komentar'].tolist()

# BERTopic per kategori sentimen
model_positif = BERTopic(embedding_model=embedding_model, verbose=True)
topics_positif, probs_positif = model_positif.fit_transform(komentar_positif)

model_netral = BERTopic(embedding_model=embedding_model, verbose=True)
topics_netral, probs_netral = model_netral.fit_transform(komentar_netral)

model_negatif = BERTopic(embedding_model=embedding_model, verbose=True)
topics_negatif, probs_negatif = model_negatif.fit_transform(komentar_negatif)

In [ ]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize

# Fungsi untuk menghitung coherence score dari hasil model BERTopic
def hitung_coherence_score(model, komentar):
    # Dapatkan daftar topik (top 10 kata per topik)
    topik_kata = model.get_topics()
    topik_kata_list = [[kata for kata, _ in topik_kata[i]] for i in topik_kata]

    # Preprocessing komentar
    tokenized_docs = [word_tokenize(doc.lower()) for doc in komentar]

    # Buat dictionary dan corpus dari data yang sudah di-tokenize
    dictionary = Dictionary(tokenized_docs)
    corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

    # Hitung coherence score pakai 'c_v' (bisa juga 'u_mass', 'c_uci', dll)
    coherence_model = CoherenceModel(
        topics=topik_kata_list,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    return coherence_model.get_coherence()

In [ ]:
# Hitung coherence score per model
coherence_positif = hitung_coherence_score(model_positif, komentar_positif)
coherence_netral  = hitung_coherence_score(model_netral, komentar_netral)
coherence_negatif = hitung_coherence_score(model_negatif, komentar_negatif)

# Tampilkan hasilnya
print("Coherence Score Positif:", coherence_positif)
print("Coherence Score Netral :", coherence_netral)
print("Coherence Score Negatif:", coherence_negatif)

In [ ]:
print("Topik Positif:")
model_positif.get_topic_info()

In [ ]:
model_positif.get_topic_info().to_csv('topik positif.csv', index=False)

In [ ]:
print("Topik Netral:")
model_netral.get_topic_info()

In [ ]:
model_netral.get_topic_info().to_csv('topik netral.csv', index=False)

In [ ]:
print("Topik Negatif:")
model_negatif.get_topic_info()

In [ ]:
model_negatif.get_topic_info().to_csv("topik_negatif.csv", index=False)

In [ ]:
topik_positif = pd.read_csv('/kaggle/working/topik positif.csv')
topik_positif

In [ ]:
# Misal kamu hanya ingin ambil topik 1, 2, dan 4
topik_penting = [0, 1, 6, 8, 17]

# Filter hanya baris dengan topik tersebut
topikpos_terpilih = topik_positif[topik_positif['Topic'].isin(topik_penting)]

# Tampilkan hasilnya
display(topikpos_terpilih)

In [ ]:
# Mapping topik ID ke nama baru
topik_rename_map = {
    0: "Pelayanan Cepat dan Ramah",
    1: "CS Sangat Membantu",
    6: "Tempat Nyaman",
    8: "Security Ramah",
    17: "Parkir Luas"
}

# Ganti nama topik berdasarkan mapping
topikpos_terpilih['Name'] = topikpos_terpilih.apply(
    lambda row: topik_rename_map.get(row['Topic'], row['Name']),
    axis=1
)

# Tampilkan
display(topikpos_terpilih)

In [ ]:
topikpos_terpilih.to_csv('topik positif label.csv', index=False)

In [ ]:
topik_negatif = pd.read_csv('/kaggle/working/topik_negatif.csv')
topik_negatif

In [ ]:
# Misal kamu hanya ingin ambil topik 1, 2, dan 4
topik_penting = [0, 1, 2, 6]

# Filter hanya baris dengan topik tersebut
topikneg_terpilih = topik_negatif[topik_negatif['Topic'].isin(topik_penting)]

# Tampilkan hasilnya
display(topikneg_terpilih)

In [ ]:
# Mapping topik ID ke nama baru
topik_rename_map = {
    0: "Pelayanan CS Lama",
    1: "Antrean Diselak Nasabah Premium",
    2: "Pelayanan Buruk",
    6: "Lokasi Tidak Sesuai di Map"
}

# Ganti nama topik berdasarkan mapping
topikneg_terpilih['Name'] = topikneg_terpilih.apply(
    lambda row: topik_rename_map.get(row['Topic'], row['Name']),
    axis=1
)

# Tampilkan
display(topikneg_terpilih)

In [ ]:
topikneg_terpilih.to_csv('topik negatif label.csv', index=False)

In [ ]:
topik_netral = pd.read_csv('/kaggle/working/topik netral.csv')
topik_netral

In [ ]:
# Mapping topik ID ke nama baru
topik_rename_map = {
    -1: "Mixed"}

# Ganti nama topik berdasarkan mapping
topik_netral['Name'] = topik_netral.apply(
    lambda row: topik_rename_map.get(row['Topic'], row['Name']),
    axis=1
)

# Tampilkan
display(topik_netral)

In [ ]:
topik_netral.to_csv('topik netral label.csv', index=False)

# Visualisasi Hasil TM

## Positif

In [ ]:
topik_positif = pd.read_csv('/kaggle/working/topik positif label.csv')
topik_positif

In [ ]:
# Urutkan dari kecil ke besar (agar bar horizontal rapi)
df_sorted = topik_positif.sort_values("Count", ascending=True)

plt.figure(figsize=(10, 5))
ax = sns.barplot(x="Count", y="Name", data=df_sorted, color="#5cb85c")

# Judul dan label
plt.title("Top Topik Positif", fontsize=14, weight='bold')
plt.xlabel("Jumlah Komentar")
plt.ylabel("Topik")
plt.grid(axis='x', linestyle='--', alpha=0.5)

# Tambahkan angka jumlah di ujung setiap bar
for i, value in enumerate(df_sorted["Count"]):
    plt.text(
        x=value + 3,  # beri jarak dari ujung bar
        y=i,          # posisi y = indeks bar
        s=str(value),
        va='center',
        fontsize=10,
        color='white',
        weight='bold'
    )

plt.tight_layout()
plt.show()

In [ ]:
for _, row in topik_positif.iterrows():
    print(f"\n🟢 {row['Name']}")
    docs = eval(row['Representative_Docs'])
    for doc in docs[:2]:  # tampilkan 3 komentar per topik
        print(f"• {doc}")


In [ ]:
topik_negatif = pd.read_csv('/kaggle/working/topik negatif label.csv')

In [ ]:
# Urutkan topik negatif berdasarkan jumlah komentar
df_sorted = topik_negatif.sort_values("Count", ascending=True)

plt.figure(figsize=(10, 5))
ax = sns.barplot(x="Count", y="Name", data=df_sorted, color="#d9534f")

# Judul dan label
plt.title("Top Topik Negatif", fontsize=14, weight='bold')
plt.xlabel("Jumlah Komentar")
plt.ylabel("Topik")
plt.grid(axis='x', linestyle='--', alpha=0.5)

# Tambahkan angka jumlah di ujung setiap bar
for i, value in enumerate(df_sorted["Count"]):
    plt.text(
        x=value + 3,  # geser sedikit kanan dari batang
        y=i,
        s=str(value),
        va='center',
        fontsize=10,
        color='white',
        weight='bold'
    )

plt.tight_layout()
plt.show()

In [ ]:
for _, row in topik_negatif.iterrows():
    print(f"\n🟢 {row['Name']}")
    docs = eval(row['Representative_Docs'])
    for doc in docs[:2]:  # tampilkan 3 komentar per topik
        print(f"• {doc}")


In [ ]:
# Urutkan topik netral berdasarkan jumlah komentar
df_sorted = topik_netral.sort_values("Count", ascending=True)

plt.figure(figsize=(10, 5))
ax = sns.barplot(x="Count", y="Name", data=df_sorted, color="#999999")

# Judul dan label
plt.title("Top Topik Netral", fontsize=14, weight='bold')
plt.xlabel("Jumlah Komentar")
plt.ylabel("Topik")
plt.grid(axis='x', linestyle='--', alpha=0.5)

# Tambahkan angka jumlah di ujung setiap bar
for i, value in enumerate(df_sorted["Count"]):
    plt.text(
        x=value + 3,
        y=i,
        s=str(value),
        va='center',
        fontsize=10,
        color='white',
        weight='bold'
    )

plt.tight_layout()
plt.show()

In [ ]:
for _, row in topik_netral.iterrows():
    print(f"\n🟢 {row['Name']}")
    docs = eval(row['Representative_Docs'])
    for doc in docs[:2]:  # tampilkan 3 komentar per topik
        print(f"• {doc}")
